# NB10 v3 -- car-only, confidence_threshold sweep (corrected render template)

Single tile: `dop20_32_473_5525_1_he`. Car-only only (`words=["car"]`) --
v2's full-class Run B is dropped per follow-up instruction; this matches
v1's original approach.

Sweeps `confidence_threshold = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]`
(9 values). `prob_thd=0.1`, `slide_crop=1024`, `slide_stride=768` held
fixed, same as v1/v2.

**Rendering now reuses NB09's actual `to_rgb`/`render_result` functions
verbatim** (copied from `notebooks/push/nb09/NB09_zeroshot_sensitivity.ipynb`
cell 13, itself re-synced 2026-07-26 from `dummyirl/SegEarth-OV-3`'s
`segment.py`, commit `55357cf`) -- v2 had reimplemented its own simplified
render function that used a different figsize/dpi/legend layout/alpha;
this was flagged as wrong and is fixed here to match NB09's current
output-image template exactly (figsize=(10,7), dpi=300, 4-column legend
grid, alpha=0.5, 2-line meta text, no bbox_inches='tight').


## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX \u2192 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated \u2192 {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned \u2192 {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 -- Inference: car-only, confidence_threshold sweep (9 reruns)

In [ ]:
%%bash
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import sys, json, torch, torch.nn.functional as F
import numpy as np
from pathlib import Path
from PIL import Image

sys.stdout.reconfigure(line_buffering=True)

DEVICE   = "cuda"
OUT_DIR  = Path("/kaggle/working/output"); OUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = OUT_DIR / "preds"; PRED_DIR.mkdir(parents=True, exist_ok=True)
BG_IDX = 255

PROB_THD     = 0.1
SLIDE_CROP   = 1024
SLIDE_STRIDE = 768
CONF_THD_SWEEP = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]

STEM = "dop20_32_473_5525_1_he"
CAR_ONLY_WORDS = ["car"]
CAR_ONLY_DISPLAY = ["car"]
CAR_ONLY_COLORS = [[255, 255, 0]]

def find_tile(stem):
    hits = sorted(Path("/kaggle/input").rglob(f"{stem}.jpg"))
    return hits[0] if hits else None

img_path = find_tile(STEM)
print(f"Resolved {STEM} -> {img_path}", flush=True)
if img_path is None:
    print("ERROR: tile not found under /kaggle/input.", flush=True)
    raise SystemExit(1)

from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("Loading SAM3...", flush=True)
model = build_sam3_image_model(
    bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE)
model.eval()
for p in model.parameters(): p.requires_grad = False
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

def make_processor(conf_thd):
    return Sam3Processor(model, confidence_threshold=conf_thd, device=DEVICE)

def cache_text(processor, words):
    cache = []
    with torch.no_grad():
        for word in words:
            te = model.backbone.forward_text([word], device=DEVICE)
            cache.append({k: v.cpu() for k, v in te.items()})
    return cache

def collect_class_scores(processor, state, h, w, te_cache, n_classes, device):
    logits = torch.zeros((n_classes, h, w), device=device)
    for cls_idx, te_cpu in enumerate(te_cache):
        processor.reset_all_prompts(state)
        for k, v in te_cpu.items(): state["backbone_out"][k] = v.to(device)
        state["geometric_prompt"] = model._get_dummy_prompt()
        processor._forward_grounding(state)
        scores = torch.zeros((h, w), device=device)
        if state.get("masks_logits") is not None and state["masks_logits"].shape[0] > 0:
            for i in range(state["masks_logits"].shape[0]):
                il = state["masks_logits"][i].squeeze()
                if il.shape != (h, w):
                    il = F.interpolate(il.view(1,1,*il.shape), size=(h,w),
                                       mode="bilinear", align_corners=False).squeeze()
                scores = torch.max(scores, il * state["object_score"][i])
        sem = state["semantic_mask_logits"].squeeze()
        if sem.shape != (h, w):
            sem = F.interpolate(sem.view(1,1,*sem.shape), size=(h,w),
                                mode="bilinear", align_corners=False).squeeze()
        scores = torch.max(scores, sem) * state["presence_score"]
        logits[cls_idx] = torch.max(logits[cls_idx], scores)
    return logits

def make_gaussian_kernel(h, w, dev):
    sy, sx = h/4.0, w/4.0
    y = torch.arange(h, device=dev).float() - (h-1)/2.0
    x = torch.arange(w, device=dev).float() - (w-1)/2.0
    return torch.exp(-y[:,None]**2/(2*sy**2)) * torch.exp(-x[None,:]**2/(2*sx**2))

def run_sliding_window(img_arr, words, processor, crop_size, stride):
    te_cache = cache_text(processor, words)
    n_cls = len(words)
    H_full, W_full = img_arr.shape[:2]

    h_grids = max(H_full - crop_size + stride - 1, 0) // stride + 1
    w_grids = max(W_full - crop_size + stride - 1, 0) // stride + 1
    total = h_grids * w_grids

    gauss_k = make_gaussian_kernel(crop_size, crop_size, DEVICE)
    acc     = torch.zeros(n_cls, H_full, W_full, device=DEVICE)
    wt_mat  = torch.zeros(H_full, W_full, device=DEVICE)

    for hi in range(h_grids):
        for wi in range(w_grids):
            y1 = hi*stride;  x1 = wi*stride
            y2 = min(y1+crop_size, H_full);  x2 = min(x1+crop_size, W_full)
            y1 = max(y2-crop_size, 0);       x1 = max(x2-crop_size, 0)

            crop_pil = Image.fromarray(img_arr[y1:y2, x1:x2])
            h_c, w_c = y2-y1, x2-x1

            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                state = processor.set_image(crop_pil)
                l = collect_class_scores(processor, state, h_c, w_c, te_cache, n_cls, DEVICE).float()

            g = gauss_k[:h_c, :w_c]
            acc[:, y1:y2, x1:x2] += l * g.unsqueeze(0)
            wt_mat[y1:y2, x1:x2] += g

            done = hi*w_grids + wi + 1
            print(f"    crop {done}/{total}", flush=True)

    return acc / wt_mat.unsqueeze(0)

def finalize(prob_map, prob_thd, bg_idx=BG_IDX):
    seg = prob_map.argmax(0)
    seg[prob_map.max(0)[0] < prob_thd] = bg_idx
    return seg.cpu().numpy()

manifest = []

def save_pred(mode, conf_thd, seg):
    tag = f"{mode}_conf{conf_thd}"
    np.save(str(PRED_DIR / f"{STEM}_{tag}.npy"), seg.astype(np.uint8))
    manifest.append(dict(stem=STEM, mode=mode, tag=tag, prob_thd=PROB_THD, conf_thd=conf_thd,
                          slide_stride=SLIDE_STRIDE, slide_crop=SLIDE_CROP))
    print(f"  Saved pred: {tag}", flush=True)

img_arr = np.array(Image.open(img_path).convert("RGB"))
img_size = (img_arr.shape[1], img_arr.shape[0])

print("\n=== Car-only sweep ===", flush=True)
for ct in CONF_THD_SWEEP:
    print(f"  [confidence_threshold={ct}] car-only", flush=True)
    proc = make_processor(ct)
    logits = run_sliding_window(img_arr, CAR_ONLY_WORDS, proc, SLIDE_CROP, SLIDE_STRIDE)
    seg = finalize(logits, PROB_THD)
    save_pred("caronly", ct, seg)

(PRED_DIR / f"{STEM}_meta.json").write_text(json.dumps(dict(
    img_size=img_size, display=CAR_ONLY_DISPLAY, colors=CAR_ONLY_COLORS)))

(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"\nWrote manifest.json with {len(manifest)} entries", flush=True)
PYEOF


## 4 -- Metrics + rendering (GPU-free)

Renders each threshold using NB09's actual re-synced render_result
template (figsize=(10,7), dpi=300, 4-column legend grid, alpha=0.5,
2-line meta text) plus the car pixel/blob-count trend plot.


In [ ]:
import json
import numpy as np
from pathlib import Path
from PIL import Image
from scipy.ndimage import label
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OUT_DIR  = Path("/kaggle/working/output")
PRED_DIR = OUT_DIR / "preds"
BG_IDX = 255

def find_tile(stem):
    hits = sorted(Path("/kaggle/input").rglob(f"{stem}.jpg"))
    return hits[0] if hits else None

# to_rgb / render_result copied verbatim from
# notebooks/push/nb09/NB09_zeroshot_sensitivity.ipynb cell 13 (re-synced
# 2026-07-26 from dummyirl/SegEarth-OV-3's segment.py, commit 55357cf).
# Do not reformat -- this must match NB09's current output-image template
# exactly (figsize, dpi, legend grid, alpha, meta text layout).

def to_rgb(seg, color_map, bg_idx=BG_IDX):
    out = np.zeros((*seg.shape, 3), dtype=np.uint8)
    safe = np.where(seg == bg_idx, 0, seg)
    out[:] = color_map[np.clip(safe, 0, len(color_map)-1)]
    out[seg == bg_idx] = [30, 30, 30]
    return out

def render_result(stem, img_arr, seg, color_map, display_labels, out_path,
                   img_size, prob_thd, conf_thd, slide_stride, slide_crop, alpha=0.5):
    fig, ax = plt.subplots(1, 2, figsize=(10, 7), dpi=300)
    fig.subplots_adjust(wspace=0)

    ax[0].imshow(img_arr)
    ax[0].axis('off')
    ax[0].set_title(f"{stem}.jpg", fontsize=10, fontweight='bold')

    ax[1].imshow(img_arr)
    ax[1].imshow(to_rgb(seg, color_map), alpha=alpha)
    ax[1].axis('off')
    ax[1].set_title(f'Segmentation Result (α={alpha})', fontsize=10, fontweight='bold')

    fig.tight_layout(rect=[0, 0.15, 1, 1])

    legend_elements = []
    for class_name, color in zip(display_labels, color_map):
        legend_elements.append(Patch(facecolor=color / 255.0, edgecolor='black', label=class_name))

    fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0.1),
               frameon=False, ncol=min(4, len(display_labels)), prop={'size': 9, 'weight': 'bold'})

    meta_text = (f"img_size = {img_size[0]}x{img_size[1]}    "
                 f"prob_thd = {prob_thd}    "
                 f"conf_thd = {conf_thd}\n"
                 f"slide_stride = {slide_stride}    "
                 f"slide_crop = {slide_crop}")
    fig.text(0.5, 0.025, meta_text, ha='center', va='bottom', fontsize=10, family='monospace')

    fig.savefig(str(out_path))
    plt.close(fig)
    print(f"  Rendered: {out_path.name}", flush=True)

manifest = json.loads((OUT_DIR / "manifest.json").read_text())
STEM = manifest[0]["stem"]
meta = json.loads((PRED_DIR / f"{STEM}_meta.json").read_text())
DISPLAY_LABELS = meta["display"]
COLOR_MAP = np.array(meta["colors"], dtype=np.uint8)
img_size = tuple(meta["img_size"])

img_arr = np.array(Image.open(find_tile(STEM)).convert("RGB"))

caronly_rows = []
for entry in manifest:
    seg = np.load(str(PRED_DIR / f"{STEM}_{entry['tag']}.npy"))
    car_mask = seg == 0
    pix = int(car_mask.sum())
    _, blobs = label(car_mask)
    caronly_rows.append((entry["conf_thd"], pix, blobs))

    render_result(STEM, img_arr, seg, COLOR_MAP, DISPLAY_LABELS,
                  OUT_DIR / f"{STEM}_{entry['tag']}.png", img_size,
                  entry["prob_thd"], entry["conf_thd"],
                  entry["slide_stride"], entry["slide_crop"])

caronly_rows.sort()
print("Car-only run:", caronly_rows)

cts, pix, blobs = zip(*caronly_rows)
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(cts, pix, "o-", color="tab:blue", label="car pixel count")
ax1.set_xlabel("confidence_threshold")
ax1.set_ylabel("car pixel count", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax2 = ax1.twinx()
ax2.plot(cts, blobs, "s--", color="tab:red", label="car blob count")
ax2.set_ylabel("car blob count", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")
plt.title(f"{STEM}: car detection vs confidence_threshold")
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_trend.png"), dpi=150)
plt.close(fig)

print(f"\nRendered {len(manifest)} images + 1 trend plot for {STEM}.", flush=True)
